# PostgreSQL + Pandas & Data Analysis Practice

## Loading Data from PostgreSQL

In [46]:
# Import libraries for database analysis

import pandas as pd
import psycopg

In [47]:
# Connect to the chinook database.

conn = psycopg.connect(
    host = 'localhost',
    port = 5432,
    dbname = 'chinook',
    user = 'postgres',
    password = '231165'
)

### Load a Table into Pandas

In [54]:
query = '''
SELECT
    track_id,
    name,
    milliseconds,
    album_id,
    genre_id,
    unit_price
FROM track
'''

df = pd.read_sql_query(query, conn)

df

C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_17216\842059613.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,track_id,name,milliseconds,album_id,genre_id,unit_price
0,1,For Those About To Rock (We Salute You),343719,1,1,0.99
1,2,Balls to the Wall,342562,2,1,0.99
2,3,Fast As a Shark,230619,3,1,0.99
3,4,Restless and Wild,252051,3,1,0.99
4,5,Princess of the Dawn,375418,3,1,0.99
...,...,...,...,...,...,...
3498,3499,Pini Di Roma (Pinien Von Rom) \ I Pini Della V...,286741,343,24,0.99
3499,3500,"String Quartet No. 12 in C Minor, D. 703 ""Quar...",139200,344,24,0.99
3500,3501,"L'orfeo, Act 3, Sinfonia (Orchestra)",66639,345,24,0.99
3501,3502,"Quintet for Horn, Violin, 2 Violas, and Cello ...",221331,346,24,0.99


## SQL Queries with Pandas

### Filtering Data in SQL

In [55]:
# Retrieve tracks longer than five minutes

query = """
SELECT
    name,
    milliseconds,
    unit_price
FROM track
WHERE milliseconds > 300000
ORDER BY milliseconds DESC;
"""

df_long_tracks = pd.read_sql_query(query, conn)

df_long_tracks.head(10)

C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_17216\4075138028.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_long_tracks = pd.read_sql_query(query, conn)


,name,milliseconds,unit_price
0,Occupation / Precipice,5286953,1.99
1,Through a Looking Glass,5088838,1.99
2,"Greetings from Earth, Pt. 1",2960293,1.99
3,The Man With Nine Lives,2956998,1.99
4,"Battlestar Galactica, Pt. 2",2956081,1.99
5,"Battlestar Galactica, Pt. 1",2952702,1.99
6,Murder On the Rising Star,2935894,1.99
7,"Battlestar Galactica, Pt. 3",2927802,1.99
8,Take the Celestra,2927677,1.99
9,Fire In Space,2926593,1.99


### Aggregation in SQL

In [56]:
# Calculate revenue by country

query = """
SELECT
    billing_country,
    COUNT(*) AS invoice_count,
    SUM(total) AS total_revenue,
    AVG(total) AS avg_invoice
FROM invoice
GROUP BY billing_country
ORDER BY total_revenue DESC;
"""

df_country_sales = pd.read_sql_query(query, conn)

df_country_sales.head(10)

C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_17216\27421837.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_country_sales = pd.read_sql_query(query, conn)


,billing_country,invoice_count,total_revenue,avg_invoice
0,USA,91,523.06,5.747912
1,Canada,56,303.96,5.427857
2,France,35,195.10,5.574286
3,Brazil,35,190.10,5.431429
4,Germany,28,156.48,5.588571
5,United Kingdom,21,112.86,5.374286
6,Czech Republic,14,90.24,6.445714
7,Portugal,14,77.24,5.517143
8,India,13,75.26,5.789231
9,Chile,7,46.62,6.660000


## Data Analysis with DataFrames

### Descriptive Statistics

In [57]:
# Summarize track duration and price

df[["milliseconds", "unit_price"]].describe()

,milliseconds,unit_price
count,3.503000e+03,3503.000000
mean,3.935992e+05,1.050805
std,5.350054e+05,0.239006
min,1.071000e+03,0.990000
25%,2.072810e+05,0.990000
50%,2.556340e+05,0.990000
75%,3.216450e+05,0.990000
max,5.286953e+06,1.990000


### Creating New Columns

In [58]:
# Convert track duration from milliseconds to minutes

df["duration_minutes"] = (
    df["milliseconds"] / 60000
)

df[["name", "milliseconds", "duration_minutes"]].head()

,name,milliseconds,duration_minutes
0,For Those About To Rock (We Salute You),343719,5.728650
1,Balls to the Wall,342562,5.709367
2,Fast As a Shark,230619,3.843650
3,Restless and Wild,252051,4.200850
4,Princess of the Dawn,375418,6.256967


### GroupBY Analysis

In [59]:
# Calculate average duration by genre

df_genre_duration = (
    df
    .groupby("genre_id")["duration_minutes"]
    .mean()
    .sort_values(ascending=False)
)

df_genre_duration.head(10)

genre_id
20    48.529717
18    43.759151
21    42.921396
19    35.750684
22    26.421062
3      5.162491
15     5.049763
13     4.957549
24     4.897793
2      4.862590
Name: duration_minutes, dtype: float64

## Sales Analysis Project

### Customer Revenue Analysis

In [60]:
# Calculate total spending for each customer

query = """
SELECT
    c.customer_id,
    c.first_name,
    c.last_name,
    c.country,
    COUNT(i.invoice_id) AS invoice_count,
    COALESCE(SUM(i.total), 0) AS total_spending
FROM customer AS c
LEFT JOIN invoice AS i
    ON c.customer_id = i.customer_id
GROUP BY
    c.customer_id,
    c.first_name,
    c.last_name,
    c.country
ORDER BY total_spending DESC;
"""

df_customers = pd.read_sql_query(query, conn)

df_customers.head(10)

C:\Users\Lighthouse\AppData\Local\Temp\ipykernel_17216\1565387676.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_customers = pd.read_sql_query(query, conn)


,customer_id,first_name,last_name,country,invoice_count,total_spending
0,6,Helena,Holý,Czech Republic,7,49.62
1,26,Richard,Cunningham,USA,7,47.62
2,57,Luis,Rojas,Chile,7,46.62
3,45,Ladislav,Kovács,Hungary,7,45.62
4,46,Hugh,O'Reilly,Ireland,7,45.62
5,28,Julia,Barnett,USA,7,43.62
6,37,Fynn,Zimmermann,Germany,7,43.62
7,24,Frank,Ralston,USA,7,43.62
8,7,Astrid,Gruber,Austria,7,42.62
9,25,Victor,Stevens,USA,7,42.62


### Identify Top Customers

In [61]:
# Select the ten highest-spending customers

top_customers = df_customers.nlargest(
    10,
    "total_spending"
)

top_customers[
    ["first_name", "last_name", "country", "total_spending"]
]

,first_name,last_name,country,total_spending
0,Helena,Holý,Czech Republic,49.62
1,Richard,Cunningham,USA,47.62
2,Luis,Rojas,Chile,46.62
3,Ladislav,Kovács,Hungary,45.62
4,Hugh,O'Reilly,Ireland,45.62
5,Julia,Barnett,USA,43.62
6,Fynn,Zimmermann,Germany,43.62
7,Frank,Ralston,USA,43.62
8,Astrid,Gruber,Austria,42.62
9,Victor,Stevens,USA,42.62
